In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
# data
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

# tensor
import torch

# preprocessing
from skimage import io
import requests
from PIL import Image
from io import BytesIO

# model
from transformers import ViTImageProcessor, ViTModel, ViTConfig

## Utils

In [ ]:
# 이미지 로딩 & 전처리 함수 (에러 인덱스 기록 포함)
def load_and_preprocess_vit(url, idx=None, target_size=(224,224)):

    try:
        response = requests.get(url, stream=True, timeout=180)
        image = Image.open(BytesIO(response.content)).convert("RGB").resize(target_size)
        return image  # ViT에서는 PIL.Image로 반환

    except Exception as e:
        print(f"[{idx}] Error fetching image for ViT: {e}")
        if idx is not None:
            error_indices.append(idx)
        # placeholder 이미지 생성 (검은색)
        image = Image.fromarray(np.zeros((*target_size, 3), dtype=np.uint8), "RGB")
        return image

In [ ]:
def predict_batch_vit(url_list, index_list, batch_size=8, model=None, image_processor=None, device='cpu'):

    # 1. 이미지 로드 (PIL.Image 객체 리스트)
    images = [load_and_preprocess_vit(url, idx) for url, idx in zip(url_list, index_list)]

    # 2. ViT 전처리 (pixel_values: (B, 3, 224, 224))
    inputs = image_processor(images, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    all_embeddings = []
    for i in range(0, len(images), batch_size):
        batch_pixels = pixel_values[i:i+batch_size]
        with torch.no_grad():
            outputs = model(batch_pixels)
            # ViTModel은 pooler_output 제공 (B, 768)
            embeddings = outputs.pooler_output.cpu().numpy()
        all_embeddings.extend(embeddings)
    return all_embeddings

## Load Dataset

In [ ]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_RQ_v1.pkl')
print(df.shape)
df.head()

(66718, 6)


,user_id,asin,text,rating,title_y,medium_image_url
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,https://images-na.ssl-images-amazon.com/images...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,https://images-na.ssl-images-amazon.com/images...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,https://images-na.ssl-images-amazon.com/images...
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,https://images-na.ssl-images-amazon.com/images...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,https://images-na.ssl-images-amazon.com/images...


## Extract Vision Transformer(ViT) Image Embedding

In [6]:
# pytorch cuda 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [7]:
# 사전 학습된 모델 불러오기
model_name = 'google/vit-base-patch16-224-in21k'
image_processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTModel.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [ ]:
# model.trainable = False -> tensorflow
# 모델의 모든 파라미터 동결
for param in model.parameters():
    param.requires_grad = False

In [9]:
# 모델을 평가 모드로 설정
model.eval()
model.to(device)

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [ ]:
# Apply the function to each row
batch_size = 8
all_image_urls = df['medium_image_url'].tolist()
all_indices = df.index.tolist()  # 실제 DataFrame index

# 에러 인덱스
error_indices = []

# 전체 추출 루프
results = []
for i in tqdm(range(0, len(all_image_urls), batch_size)):
    batch_urls = all_image_urls[i:i+batch_size]
    batch_indices = all_indices[i:i+batch_size]
    preds = predict_batch_vit(
        batch_urls, batch_indices, batch_size=len(batch_urls),
        model=model, image_processor=image_processor, device=device
    )
    results.extend(preds)

embedding = pd.DataFrame({'ViT_emb_768': [e for e in results]})
df = pd.concat([df, embedding], axis=1)

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_RQ_v2.pkl')

In [ ]:
# error_indices에 오류난 데이터 인덱스가 다 들어있음
# 해결 완료
print(f"\n불러오지 못한 이미지 인덱스: {error_indices}")
print(f"\n총 오류 개수: {len(error_indices)}")
print("\n실패한 url:")

for idx in error_indices:
    print(df.loc[idx, 'medium_image_url'])

# 행 인덱스와 컬럼명을 이용해 직접 할당
# df.at[40283, 'ViT_emb_768'] = results[0]


불러오지 못한 이미지 인덱스: [40283]

총 오류 개수: 1

실패한 url:
https://m.media-amazon.com/images/I/610b1L8AX9L._SL800_.jpg


In [ ]:
# error embedding 확인
fail_indices = [i for i, e in enumerate(results) if np.allclose(e, 0)]
print(f"임베딩이 0으로만 채워진 샘플 index: {fail_indices}")

임베딩이 0으로만 채워진 샘플 index: []
